# Multi-Agent Stock Analyst with GLM-5.3-Flash and CrewAI

This notebook coordinates four specialists:

```text
Fundamentals ----\
Technicals -------+--> Portfolio Manager --> BUY / HOLD / SELL
Recent news -----/
```

Financial facts come from Finnhub, indicator calculations run in Python, current news comes from Tavily, and GLM-5.3-Flash interprets the evidence.

> **Educational research only:** this notebook does not provide financial advice. API results and model output can be incomplete or wrong.

## 1. Setup

The environment variables have **already been added**. This notebook reads them from the environment, validates that they exist, and never displays their values. For a fresh Python environment, install the pinned dependencies with:

```python
%pip install -r requirements.txt
```

In [1]:
import os
import re
import time

import finnhub
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient

from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool
from crewai_tools import TavilySearchTool

load_dotenv()  # Does not overwrite variables already present in the environment.
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

REQUIRED_ENV_VARS = ("ZAI_API_KEY", "FINNHUB_API_KEY", "TAVILY_API_KEY")
missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {', '.join(missing)}")

TICKER = os.getenv("STOCK_TICKER", "NVDA").strip().upper()
if not re.fullmatch(r"[A-Z][A-Z0-9.-]{0,9}", TICKER):
    raise ValueError(f"Invalid stock ticker: {TICKER!r}")

print("All required environment variables are already configured.")
print("Analyzing:", TICKER)

All required environment variables are already configured.
Analyzing: NVDA


## 2. Initialize and smoke-test the live services

These small calls fail early if a credential or endpoint is unavailable. Only status information is printed; secrets are never included in notebook output.

In [2]:
finnhub_client = finnhub.Client(api_key=os.environ["FINNHUB_API_KEY"])
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

llm = LLM(
    model="openai/glm-5.3-flash",
    api_key=os.environ["ZAI_API_KEY"],
    base_url="https://api.z.ai/api/paas/v4/",
    temperature=0.1,
)

quote_smoke = finnhub_client.quote(TICKER)
if not quote_smoke or quote_smoke.get("c") in (None, 0):
    raise RuntimeError(f"Finnhub returned no live quote for {TICKER}.")

search_smoke = tavily_client.search(
    f"{TICKER} company latest news", search_depth="basic", max_results=1
)
if not search_smoke.get("results"):
    raise RuntimeError("Tavily returned no smoke-test result.")

zai_client = OpenAI(
    api_key=os.environ["ZAI_API_KEY"],
    base_url="https://api.z.ai/api/paas/v4/",
)
zai_smoke = zai_client.chat.completions.create(
    model="glm-5.3-flash",
    messages=[{"role": "user", "content": "Reply with exactly OK."}],
    temperature=0,
    max_tokens=128,
)
if zai_smoke.choices[0].message.content.strip() != "OK":
    raise RuntimeError("GLM-5.3-Flash did not pass the smoke test.")

print("Live service checks passed: Finnhub, Tavily, and Z.ai/GLM.")

Live service checks passed: Finnhub, Tavily, and Z.ai/GLM.


## 3. Market-data functions and indicators

Finnhub quote and fundamental endpoints work on the configured account. Daily candles may require a different Finnhub plan, so `get_price_history` tries Finnhub first and transparently falls back to Yahoo's public chart response if Finnhub denies candle access. This fallback uses no extra credential and is labeled in the technical report.

In [3]:
HTTP = requests.Session()
HTTP.headers.update({"User-Agent": "glm-stock-swarm/1.0"})

def _finnhub_history(ticker: str, days: int) -> pd.DataFrame:
    end = int(time.time())
    start = end - days * 24 * 60 * 60
    response = HTTP.get(
        "https://finnhub.io/api/v1/stock/candle",
        params={
            "symbol": ticker, "resolution": "D",
            "from": start, "to": end,
            "token": os.environ["FINNHUB_API_KEY"],
        },
        timeout=30,
    )
    if response.status_code != 200:
        return pd.DataFrame()
    data = response.json()
    if data.get("s") != "ok":
        return pd.DataFrame()
    frame = pd.DataFrame({
        "date": pd.to_datetime(data["t"], unit="s", utc=True).tz_localize(None),
        "open": data["o"], "high": data["h"], "low": data["l"],
        "close": data["c"], "volume": data["v"],
    })
    frame.attrs["source"] = "Finnhub daily candles"
    return frame

def _yahoo_history(ticker: str, days: int) -> pd.DataFrame:
    end = int(time.time())
    start = end - days * 24 * 60 * 60
    response = HTTP.get(
        f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}",
        params={"period1": start, "period2": end, "interval": "1d", "events": "history"},
        timeout=30,
    )
    response.raise_for_status()
    result = response.json()["chart"]["result"][0]
    values = result["indicators"]["quote"][0]
    frame = pd.DataFrame({
        "date": pd.to_datetime(result["timestamp"], unit="s", utc=True).tz_localize(None),
        "open": values["open"], "high": values["high"], "low": values["low"],
        "close": values["close"], "volume": values["volume"],
    }).dropna(subset=["close"]).sort_values("date").reset_index(drop=True)
    frame.attrs["source"] = "Yahoo public chart fallback (Finnhub candles unavailable)"
    return frame

def get_price_history(ticker: str, days: int = 450) -> pd.DataFrame:
    ticker = ticker.strip().upper()
    frame = _finnhub_history(ticker, days)
    if frame.empty:
        frame = _yahoo_history(ticker, days)
    return frame

def add_indicators(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    result = frame.copy()
    result.attrs.update(frame.attrs)
    result["SMA20"] = result["close"].rolling(20).mean()
    result["SMA50"] = result["close"].rolling(50).mean()
    result["SMA200"] = result["close"].rolling(200).mean()
    delta = result["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    result["RSI14"] = 100 - 100 / (1 + avg_gain / avg_loss.replace(0, np.nan))
    result["EMA12"] = result["close"].ewm(span=12, adjust=False).mean()
    result["EMA26"] = result["close"].ewm(span=26, adjust=False).mean()
    result["MACD"] = result["EMA12"] - result["EMA26"]
    return result

history_smoke = add_indicators(get_price_history(TICKER))
if len(history_smoke) < 200:
    raise RuntimeError(f"Need at least 200 daily observations; received {len(history_smoke)}.")
print(f"Price history passed: {len(history_smoke)} rows from {history_smoke.attrs['source']}.")
history_smoke.tail(3)[["date", "close", "SMA20", "SMA50", "SMA200", "RSI14", "MACD"]]

Price history passed: 310 rows from Yahoo public chart fallback (Finnhub candles unavailable).


,date,close,SMA20,SMA50,SMA200,RSI14,MACD
307,2026-08-25 13:30:00,213.050003,214.578001,207.806399,195.42565,49.605429,2.193667
308,2026-08-26 13:30:00,209.660004,215.560501,207.750599,195.53355,46.294616,1.543117
309,2026-08-27 13:30:00,227.979996,217.207501,208.161999,195.73270,61.319420,2.477265


## 4. CrewAI tools

The model interprets only data returned by these tools; it is instructed not to invent missing values.

In [4]:
def _number(value, decimals: int = 2) -> str:
    if value is None or pd.isna(value):
        return "N/A"
    return f"{float(value):,.{decimals}f}"

@tool("Get Stock Fundamentals")
def get_fundamentals(ticker: str) -> str:
    """Get current price and company fundamentals from Finnhub for one ticker."""
    ticker = ticker.strip().upper()
    quote = finnhub_client.quote(ticker)
    metrics = finnhub_client.company_basic_financials(ticker, "all")
    m = metrics.get("metric", {})
    return f"""Ticker: {ticker}
Source: Finnhub
Current price: {_number(quote.get('c'))}
Previous close: {_number(quote.get('pc'))}
Daily change %: {_number(quote.get('dp'))}
Day high / low: {_number(quote.get('h'))} / {_number(quote.get('l'))}
Market cap (USD millions): {_number(m.get('marketCapitalization'))}
52-week high / low: {_number(m.get('52WeekHigh'))} / {_number(m.get('52WeekLow'))}
Normalized annual P/E: {_number(m.get('peNormalizedAnnual'))}
Annual P/B: {_number(m.get('pbAnnual'))}
Annual P/S: {_number(m.get('psAnnual'))}
ROE TTM: {_number(m.get('roeTTM'))}
Net margin TTM: {_number(m.get('netProfitMarginTTM'))}
Revenue growth TTM YoY: {_number(m.get('revenueGrowthTTMYoy'))}
EPS growth TTM YoY: {_number(m.get('epsGrowthTTMYoy'))}
Annual debt/equity: {_number(m.get('totalDebt/totalEquityAnnual'))}"""

@tool("Analyze Stock Technicals")
def get_technicals(ticker: str) -> str:
    """Calculate price trends, returns, moving averages, RSI, and MACD for one ticker."""
    ticker = ticker.strip().upper()
    frame = add_indicators(get_price_history(ticker))
    if len(frame) < 200:
        return f"Technical data unavailable for {ticker}: fewer than 200 observations."
    latest = frame.iloc[-1]
    return f"""Ticker: {ticker}
Source: {frame.attrs.get('source', 'unknown')}
Last market date: {latest['date'].date()}
Close: {_number(latest['close'])}
SMA20 / SMA50 / SMA200: {_number(latest['SMA20'])} / {_number(latest['SMA50'])} / {_number(latest['SMA200'])}
RSI14: {_number(latest['RSI14'])}
MACD: {_number(latest['MACD'])}
20-day return: {_number((latest['close'] / frame['close'].iloc[-20] - 1) * 100)}%
50-day return: {_number((latest['close'] / frame['close'].iloc[-50] - 1) * 100)}%
Price vs SMA20: {_number((latest['close'] / latest['SMA20'] - 1) * 100)}%
Price vs SMA50: {_number((latest['close'] / latest['SMA50'] - 1) * 100)}%
Price vs SMA200: {_number((latest['close'] / latest['SMA200'] - 1) * 100)}%"""

web_search = TavilySearchTool(
    api_key=os.environ["TAVILY_API_KEY"],
    topic="news",
    search_depth="advanced",
    days=30,
    max_results=5,
)

print(get_fundamentals.run(ticker=TICKER))
print()
print(get_technicals.run(ticker=TICKER))

Ticker: NVDA
Source: Finnhub
Current price: 227.98
Previous close: 209.66
Daily change %: 8.74
Day high / low: 230.47 / 220.90
Market cap (USD millions): 5,422,978.00
52-week high / low: 236.54 / 164.07
Normalized annual P/E: 45.17
Annual P/B: 28.81
Annual P/S: 25.11
ROE TTM: 111.66
Net margin TTM: 62.97
Revenue growth TTM YoY: 70.68
EPS growth TTM YoY: 110.34
Annual debt/equity: 0.05



Ticker: NVDA
Source: Yahoo public chart fallback (Finnhub candles unavailable)
Last market date: 2026-08-27
Close: 227.98
SMA20 / SMA50 / SMA200: 217.21 / 208.16 / 195.73
RSI14: 61.32
MACD: 2.48
20-day return: 13.56%
50-day return: 11.40%
Price vs SMA20: 4.96%
Price vs SMA50: 9.52%
Price vs SMA200: 16.48%


## 5. Agents and tasks

In [5]:
fundamental_agent = Agent(
    role="Fundamental Analyst",
    goal="Evaluate financial health, growth, profitability, valuation, and balance-sheet risk.",
    backstory="You are a careful long-term equity analyst. Use the fundamentals tool and never invent a figure. Treat N/A as missing, not as zero.",
    tools=[get_fundamentals], llm=llm, allow_delegation=False, max_iter=4, verbose=False,
)
technical_agent = Agent(
    role="Technical Analyst",
    goal="Determine whether the current price setup is bullish, bearish, or neutral.",
    backstory="You interpret indicators calculated by Python. Never guess market prices or indicator values.",
    tools=[get_technicals], llm=llm, allow_delegation=False, max_iter=4, verbose=False,
)
news_agent = Agent(
    role="Financial News Analyst",
    goal="Find material recent developments and distinguish confirmed reporting from speculation.",
    backstory="You are a skeptical financial-news researcher. Prioritize primary and reputable sources, dates, and links.",
    tools=[web_search], llm=llm, allow_delegation=False, max_iter=5, verbose=False,
)
manager_agent = Agent(
    role="Portfolio Manager",
    goal="Combine the specialist reports into a balanced, evidence-based research signal.",
    backstory="You lead an equity research team. Weigh bullish and bearish evidence, use HOLD when evidence is mixed, and never add unsupported facts.",
    llm=llm, allow_delegation=False, max_iter=4, verbose=False,
)

fundamental_task = Task(
    description=f"Analyze {TICKER}'s growth, profitability, valuation, balance sheet, and business quality. Use Get Stock Fundamentals. Return a Fundamental Score from 0-100, strongest positive, biggest risk, and note all material missing data.",
    expected_output="A concise fundamental assessment with a 0-100 score, evidence, strongest positive, biggest risk, and missing-data note.",
    agent=fundamental_agent,
)
technical_task = Task(
    description=f"Analyze {TICKER}'s technical setup. Use Analyze Stock Technicals. Consider SMA20/50/200, RSI14, MACD, recent returns, and trend. Return Technical Score 0-100, Bullish/Neutral/Bearish signal, trend, momentum, and main technical risk.",
    expected_output="A concise technical assessment with score, signal, trend, momentum, data source, and main risk.",
    agent=technical_agent,
)
news_task = Task(
    description=f"Research the most important recent news for {TICKER}, focusing on the last 30 days: earnings, guidance, analyst revisions, products, partnerships, M&A, regulation, lawsuits, management, and industry developments. Use Tavily Search. Return News Score 0-100, sentiment, positive and negative catalysts, publication dates, and source URLs. Ignore low-quality speculation.",
    expected_output="A sourced recent-news assessment with score, sentiment, catalysts, dates, and clickable URLs.",
    agent=news_agent,
)
decision_task = Task(
    description=f"Review all three specialist reports and make the final educational research assessment for {TICKER}. Return exactly one signal: BUY, HOLD, or SELL. Use HOLD when evidence is mixed or confidence is insufficient. Base every claim only on the supplied reports.",
    expected_output="Ticker; Signal; Overall Score /100; Confidence %; Fundamental, Technical, and News scores; Bull Case; Bear Case; Main Catalyst; Main Risk; Final Explanation; Sources; and an educational-not-financial-advice disclaimer.",
    agent=manager_agent,
    context=[fundamental_task, technical_task, news_task],
)

## 6. Run the stock-research crew

The workflow is sequential: the three specialists collect and interpret evidence, then the portfolio manager receives their reports as context.

In [6]:
crew = Crew(
    agents=[fundamental_agent, technical_agent, news_agent, manager_agent],
    tasks=[fundamental_task, technical_task, news_task, decision_task],
    process=Process.sequential,
    verbose=False,
    tracing=False,
)

result = await crew.kickoff_async()
print(result.raw)

# NVDA — Final Research Assessment (Educational Purposes Only)

**Ticker:** NVDA
**Signal:** **BUY**
**Overall Score:** **84 / 100**
**Confidence:** **75%**

| Component | Score |
|---|---|
| Fundamental | 84 / 100 |
| Technical | 80 / 100 |
| News (30-day) | 88 / 100 |

---

## Bull Case

- **Hyper-growth at unprecedented scale:** TTM revenue +70.68%, TTM EPS +110.34%; Q2 FY27 revenue of **$96.22B (+106% YoY, +18% QoQ)** beat consensus (~$92.2B) by ~4%, with Data Center revenue of $89.0B (+117% YoY) — per NVIDIA's primary-source earnings release.
- **Elite profitability:** 75.0% gross margin (up 2.6 pts YoY), 62.97% TTM net margin, ROE of 111.66%, operating income +124% YoY, net income $59.7B.
- **Fortress balance sheet and shareholder returns:** Debt/equity of just 0.05; ~$26.0B returned to shareholders in Q2 with **$99.0B remaining** buyback authorization.
- **Guidance power:** Q3 FY27 revenue guide of **$108B ±2%** vs. ~$104.2–104.9B consensus; **FY2028 guidance of ~70% growth vs. 